In [0]:
from pyspark.sql import functions as F
enriched_transactions = spark.table("workspace.pyspark_deep_dive.enriched_transactions")

In [0]:
print(enriched_transactions.count())

In [0]:
category_sales = (
    enriched_transactions
    .groupBy("category")
    .agg(
        F.count("transaction_id").alias("total_transactions"),
        F.sum("quantity").alias("total_quantity"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy(F.col("total_revenue").desc())
)

display(category_sales)

In [0]:
customer_metrics = (
    enriched_transactions
    .groupBy(
        "customer_id",
        "customer_name",
        "city",
        "customer_segment"
    )
    .agg(
        F.count("transaction_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.round(F.sum("total_amount"), 2).alias("total_spend"),
        F.round(F.avg("total_amount"), 2).alias("average_order_value")
    )
)

display(customer_metrics.orderBy(
    F.col("total_spend").desc()
).limit(20))

In [0]:
from pyspark.sql.window import Window

city_window = Window \
    .partitionBy("city") \
    .orderBy(F.col("total_spend").desc())

top_customers = (
    customer_metrics
    .withColumn(
        "customer_rank",
        F.row_number().over(city_window)
    )
    .filter(F.col("customer_rank") <= 3)
    .orderBy("city", "customer_rank")
)

display(top_customers)

In [0]:
city_window_rank = Window \
    .partitionBy("city") \
    .orderBy(F.col("total_spend").desc())

ranked_customers = (
    customer_metrics
    .withColumn(
        "rank",
        F.rank().over(city_window_rank)
    )
    .filter(F.col("rank") <= 3)
)

display(ranked_customers)

In [0]:
customer_order_window = Window \
    .partitionBy("customer_id") \
    .orderBy("transaction_date", "transaction_id")

customer_history = (
    enriched_transactions
    .withColumn(
        "previous_order_amount",
        F.lag("total_amount").over(customer_order_window)
    )
    .withColumn(
        "amount_difference",
        F.round(
            F.col("total_amount") - F.col("previous_order_amount"),
            2
        )
    )
)

display(
    customer_history
    .filter(F.col("previous_order_amount").isNotNull())
    .limit(20)
)

In [0]:
customer_history = (
    enriched_transactions
    .withColumn(
        "next_order_amount",
        F.lead("total_amount").over(customer_order_window)
    )
)

display(customer_history.limit(20))

In [0]:
running_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("transaction_date", "transaction_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

customer_running_spend = (
    enriched_transactions
    .withColumn(
        "cumulative_spend",
        F.round(
            F.sum("total_amount").over(running_window),
            2
        )
    )
)

display(
    customer_running_spend
    .orderBy("customer_id", "transaction_date")
    .limit(20)
)

In [0]:
classified_transactions = enriched_transactions.withColumn(
    "order_value_segment",
    F.when(F.col("total_amount") >= 5000, "High Value")
     .when(F.col("total_amount") >= 2000, "Medium Value")
     .otherwise("Low Value")
)
display(classified_transactions.limit(20))

In [0]:
date_enriched = (
    enriched_transactions
    .withColumn("year", F.year("transaction_date"))
    .withColumn("month", F.month("transaction_date"))
    .withColumn("month_name", F.date_format("transaction_date", "MMMM"))
    .withColumn("day_of_week", F.date_format("transaction_date", "EEEE"))
)

display(date_enriched.limit(20))

In [0]:
monthly_sales = (
    date_enriched
    .groupBy("year", "month", "month_name")
    .agg(
        F.count("transaction_id").alias("orders"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("total_amount"), 2).alias("revenue")
    )
    .orderBy("year", "month")
)

display(monthly_sales)